In [ ]:
# Example 1: Build a simple TypeMachine with two states + a transition.
from automat import TypeMachineBuilder
from typing import Protocol

class Light(Protocol):
    def flip(self) -> None: ...
    def query(self) -> str: ...

class Core:
    pass

builder = TypeMachineBuilder(Light, Core)
off = builder.state("off")
on = builder.state("on")
off.upon(Light.flip).to(on).returns(None)
on.upon(Light.flip).to(off).returns(None)
off.upon(Light.query).loop().returns("off")
on.upon(Light.query).loop().returns("on")
machine_factory = builder.build()
print(type(machine_factory).__name__)


In [ ]:
# Example 2: Drive the TypeMachine through state transitions and assert behavior.
from automat import TypeMachineBuilder
from typing import Protocol

class Toggle(Protocol):
    def press(self) -> None: ...
    def status(self) -> str: ...

class Ctx:
    pass

b = TypeMachineBuilder(Toggle, Ctx)
s_off = b.state("off")
s_on = b.state("on")
s_off.upon(Toggle.press).to(s_on).returns(None)
s_on.upon(Toggle.press).to(s_off).returns(None)
s_off.upon(Toggle.status).loop().returns("off")
s_on.upon(Toggle.status).loop().returns("on")
factory = b.build()
m = factory(Ctx())
assert m.status() == "off"
m.press()
assert m.status() == "on"
print("transition ok")


In [ ]:
# Example 3: Legacy MethodicalMachine API still importable + usable.
from automat import MethodicalMachine

class TrafficLight:
    _machine = MethodicalMachine()

    @_machine.state(initial=True)
    def red(self): "red state"

    @_machine.state()
    def green(self): "green state"

    @_machine.input()
    def go(self): "start moving"

    @_machine.output()
    def _say_go(self): return "going"

    red.upon(go, enter=green, outputs=[_say_go], collector=list)

light = TrafficLight()
result = light.go()
assert result == ["going"]
print("MethodicalMachine output:", result)


In [ ]:
# Example 4: TypeMachine with a data state carrying per-state attributes.
# The initial state may not require state-specific data, so enter the data
# state via a transition from a plain initial state.
from automat import TypeMachineBuilder
from dataclasses import dataclass
from typing import Protocol

class Counter(Protocol):
    def start(self) -> None: ...
    def increment(self) -> int: ...

class CoreCtx:
    pass

@dataclass
class CountData:
    n: int

cb = TypeMachineBuilder(Counter, CoreCtx)
idle = cb.state("idle")
counting = cb.state("counting", lambda machine, core: CountData(n=0))
idle.upon(Counter.start).to(counting).returns(None)
@counting.upon(Counter.increment).loop()
def _inc(machine: Counter, core: CoreCtx, data: CountData) -> int:
    data.n += 1
    return data.n
f = cb.build()
m = f(CoreCtx())
m.start()
assert m.increment() == 1
assert m.increment() == 2
print("data-state ok")


In [ ]:
# Example 5: Public API surface inspection + sanity assertions.
import automat

expected = {"TypeMachineBuilder", "MethodicalMachine", "NoTransition"}
present = {name for name in expected if hasattr(automat, name)}
missing = expected - present
print(f"automat exports present: {sorted(present)}")
print(f"missing (informational): {sorted(missing)}")

# Core builder must be present
assert hasattr(automat, "TypeMachineBuilder"), "TypeMachineBuilder missing from automat"
assert callable(automat.TypeMachineBuilder), "TypeMachineBuilder must be callable"
print("automat public API surface ok")
